In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split

# This is your Day 12 data, but with prompt/completion added for fine-tuning
ft_data = pd.DataFrame({
'prompt': [
'Write a python function to add two numbers',
'How are you doing today?',
'Debug this Python code: def add(a,b) return a+b',
'Translate "hello" to Urdu',
'Write a JavaScript for loop from 1 to 5',
'Solve the equation 2x + 5 = 15'
],
'completion': [
'def add(a, b):\n return a + b',
'I am doing well, thank you for asking!',
'Missing colon after function definition. Fix: def add(a,b): return a+b',
'ہیلو',
'for(let i=1; i<=5; i++) { console.log(i); }',
'2x = 10\nx = 5'
],
'Prompt_type': ['code', 'chat', 'code', 'chat', 'code', 'math'],
'language': ['English', 'English', 'English', 'Urdu', 'English', 'English'],
'quality_score': [4.5, 3.0, 4.8, 2.5, 4.2, 5.0],
'token_length': [120, 80, 150, 60, 110, 200]
})

df = ft_data.copy()
print("Original data shape:", df.shape)
df.head()

Original data shape: (6, 6)


,prompt,completion,Prompt_type,language,quality_score,token_length
0,Write a python function to add two numbers,"def add(a, b):\n return a + b",code,English,4.5,120
1,How are you doing today?,"I am doing well, thank you for asking!",chat,English,3.0,80
2,"Debug this Python code: def add(a,b) return a+b",Missing colon after function definition. Fix: ...,code,English,4.8,150
3,"Translate ""hello"" to Urdu",ہیلو,chat,Urdu,2.5,60
4,Write a JavaScript for loop from 1 to 5,for(let i=1; i<=5; i++) { console.log(i); },code,English,4.2,110


In [10]:
# Check bias first
print("Bias Check - Before Balancing:")
print(df.groupby('Prompt_type')['quality_score'].agg(['mean', 'count']))

# Balance by upsampling chat + math to match code count
code_df = df[df['Prompt_type'] == 'code']
chat_df = df[df['Prompt_type'] == 'chat']
math_df = df[df['Prompt_type'] == 'math']

target_count = len(code_df) # 3

chat_balanced = chat_df.sample(n=target_count, replace=True, random_state=42)
math_balanced = math_df.sample(n=target_count, replace=True, random_state=42)

df_balanced = pd.concat([code_df, chat_balanced, math_balanced])
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print("\nAfter Balancing:")
print(df_balanced['Prompt_type'].value_counts())
df_balanced

Bias Check - Before Balancing:
             mean  count
Prompt_type             
chat         2.75      2
code         4.50      3
math         5.00      1

After Balancing:
Prompt_type
math    3
code    3
chat    3
Name: count, dtype: int64


,prompt,completion,Prompt_type,language,quality_score,token_length
0,Solve the equation 2x + 5 = 15,2x = 10\nx = 5,math,English,5.0,200
1,"Debug this Python code: def add(a,b) return a+b",Missing colon after function definition. Fix: ...,code,English,4.8,150
2,How are you doing today?,"I am doing well, thank you for asking!",chat,English,3.0,80
3,Write a python function to add two numbers,"def add(a, b):\n return a + b",code,English,4.5,120
4,Solve the equation 2x + 5 = 15,2x = 10\nx = 5,math,English,5.0,200
5,Write a JavaScript for loop from 1 to 5,for(let i=1; i<=5; i++) { console.log(i); },code,English,4.2,110
6,"Translate ""hello"" to Urdu",ہیلو,chat,Urdu,2.5,60
7,How are you doing today?,"I am doing well, thank you for asking!",chat,English,3.0,80
8,Solve the equation 2x + 5 = 15,2x = 10\nx = 5,math,English,5.0,200


In [15]:
train_df, test_df = train_test_split(
    df_balanced,
    test_size=0.33, # rows in test = 1 per class
    stratify=df_balanced['Prompt_type'],
    random_state=42
)

In [17]:
print(f"Train size: {len(train_df)}")

Train size: 6


In [18]:
print(f"Test size: {len(test_df)}")

Test size: 3


In [19]:
print("n\Test distribution:")

n\Test distribution:


In [21]:
print(test_df['Prompt_type'].value_counts())

Prompt_type
code    1
chat    1
math    1
Name: count, dtype: int64


In [22]:
# Export for fine-tuning

In [26]:
train_df[['prompt',
          'completion']].to_json('train.jsonl',
orient='records',lines=True)
test_df[['prompt',
         'completion']].to_json('test.jsonl',
                                orient='records', lines=True)
print("\nFiles saved: train.jsonl, test.jsonl")


Files saved: train.jsonl, test.jsonl


In [28]:
# Check file size and preview

import os

print(f"train.jsonl: {os.path.getsize('train.jsonl')} bytes")

train.jsonl: 534 bytes


In [29]:
print(f"test.jsonl: {os.path.getsize('test.jsonl')} bytes")

test.jsonl: 313 bytes


In [30]:
# See first line of each

with open('train.jsonl') as f:
    print("\nTrain sample:", f.readline().strip())


Train sample: {"prompt":"Write a JavaScript for loop from 1 to 5","completion":"for(let i=1; i<=5; i++) { console.log(i); }"}


In [32]:
with open('test.jsonl') as f:
    print("Test sample:", f.readline().strip())

Test sample: {"prompt":"Debug this Python code: def add(a,b) return a+b","completion":"Missing colon after function definition. Fix: def add(a,b): return a+b"}
